In [ ]:
"""
05_volterra_autoregressive_lm.py / 05_volterra_autoregressive_lm.ipynb

Flagship Autoregressive Language Model: Volterra-Fredholm Spectral Attention in JAX/XLA
Evaluates continuous integral attention with Chebyshev quadrature and causal dissipative semigroups.

Hardware Target: NVIDIA A100 / GPU Accelerator via JAX and XLA
Precision: Native bfloat16 Mixed Precision
Corpus: WikiText-2 (GPT-2 BPE Tokenization, 50,257 vocabulary size)

Architecture Configuration (51.2M Parameters):
  - Embedding Dimension (d_model): 512
  - Layers (n_layers): 8
  - Attention Heads (n_heads): 8
  - Head Dimension (head_dim): 64
  - Orthogonal Chebyshev Modes (m_modes): 16
  - Context Window (block_size): 512
  - Batch Size: 32
"""

import os
import math
import time
import urllib.request
from typing import Any, Dict, Tuple

import numpy as np

# Configure XLA client allocation behavior
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import jax
import jax.numpy as jnp
from jax import random

# Compatibility layer for JAX core extensions
try:
    import jax.extend.core as jec
    if not hasattr(jax.core, "get_opaque_trace_state") and hasattr(jec, "get_opaque_trace_state"):
        jax.core.get_opaque_trace_state = jec.get_opaque_trace_state
except Exception:
    pass

import flax.linen as nn
from flax.training import train_state
import optax

# -----------------------------------------------------------------------------
# 0. Global Hardware Setup & Hyperparameters
# -----------------------------------------------------------------------------
DEVICES = jax.devices()
D_MODEL: int = 512
N_LAYERS: int = 8
N_HEADS: int = 8
HEAD_DIM: int = D_MODEL // N_HEADS  # 64
BLOCK_SIZE: int = 512
BATCH_SIZE: int = 32
M_MODES: int = 16
TOTAL_STEPS: int = 1000
EVAL_INTERVAL: int = 50
EVAL_ITERS: int = 40
LR: float = 6e-4
WEIGHT_DECAY: float = 0.1
SEED: int = 42
DTYPE: jnp.dtype = jnp.bfloat16


# -----------------------------------------------------------------------------
# 1. Dataset Ingestion Pipeline (Direct Mirrors First)
# -----------------------------------------------------------------------------
def load_wikitext_dataset() -> Tuple[np.ndarray, np.ndarray, int]:
    """
    Downloads WikiText-2 via direct raw mirrors to prevent external authentication dependencies.
    Tokenizes text using the official GPT-2 BPE tokenizer.
    """
    data_dir = "/tmp/wikitext_data"
    os.makedirs(data_dir, exist_ok=True)
    train_text, valid_text = "", ""

    mirrors = [
        ("https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/train.txt",
         "https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/valid.txt"),
        ("https://huggingface.co/datasets/ggml-org/ci/resolve/main/wikitext-2-raw/wiki.train.raw",
         "https://huggingface.co/datasets/ggml-org/ci/resolve/main/wikitext-2-raw/wiki.valid.raw"),
    ]
    headers = {"User-Agent": "Mozilla/5.0"}

    for tr_url, va_url in mirrors:
        try:
            req_tr = urllib.request.Request(tr_url, headers=headers)
            req_va = urllib.request.Request(va_url, headers=headers)
            with urllib.request.urlopen(req_tr, timeout=20) as resp:
                train_text = resp.read().decode("utf-8")
            with urllib.request.urlopen(req_va, timeout=20) as resp:
                valid_text = resp.read().decode("utf-8")
            if len(train_text) > 10000:
                break
        except Exception:
            continue

    if not train_text:
        try:
            from datasets import load_dataset
            ds = load_dataset("wikitext", "wikitext-2-raw-v1")
            train_text = "\n".join(ds["train"]["text"])
            valid_text = "\n".join(ds["validation"]["text"])
        except Exception:
            raise RuntimeError("Failed to retrieve WikiText-2 dataset.")

    try:
        import tiktoken
        tokenizer = tiktoken.get_encoding("gpt2")
        train_tokens = tokenizer.encode_ordinary(train_text)
        valid_tokens = tokenizer.encode_ordinary(valid_text)
        vocab_size = tokenizer.n_vocab
    except ImportError:
        train_tokens = list(train_text.encode("utf-8"))
        valid_tokens = list(valid_text.encode("utf-8"))
        vocab_size = 256

    train_arr = np.array(train_tokens, dtype=np.int32)
    valid_arr = np.array(valid_tokens, dtype=np.int32)
    return train_arr, valid_arr, vocab_size


TRAIN_DATA, VAL_DATA, VOCAB_SIZE = load_wikitext_dataset()


def get_batch(data_array: np.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray]:
    max_idx = len(data_array) - BLOCK_SIZE - 1
    indices = np.random.randint(0, max_idx, size=(BATCH_SIZE,))
    x = np.stack([data_array[i : i + BLOCK_SIZE] for i in indices])
    y = np.stack([data_array[i + 1 : i + BLOCK_SIZE + 1] for i in indices])
    return jnp.array(x, dtype=jnp.int32), jnp.array(y, dtype=jnp.int32)


# -----------------------------------------------------------------------------
# 2. Geometric Hyperspherical Primitives
# -----------------------------------------------------------------------------
def project_conical(x: jnp.ndarray, eps: float = 1e-7) -> jnp.ndarray:
    """
    Projects feature vectors onto the unit sphere S^{d-1} and rescales by sqrt(d):
        proj(x) = sqrt(d) * (x / ||x||_2)
    """
    scale = jnp.sqrt(float(x.shape[-1]))
    norm = jnp.linalg.norm(x.astype(jnp.float32), axis=-1, keepdims=True) + eps
    return (scale * (x.astype(jnp.float32) / norm)).astype(x.dtype)


class ConicalNorm(nn.Module):
    dim: int
    eps: float = 1e-7

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        return project_conical(x, self.eps)


def eye_initializer(rng: Any, shape: Tuple[int, int], dtype: jnp.dtype = jnp.float32) -> jnp.ndarray:
    return jnp.eye(shape[0], shape[1], dtype=dtype)


# -----------------------------------------------------------------------------
# 3. Canonical Volterra-Fredholm Spectral Attention (XLA Fused)
# -----------------------------------------------------------------------------
class ScaledVolterraAttention(nn.Module):
    """
    Continuous causal attention operator based on the Fredholm integral equation
    discretized via Gauss-Chebyshev quadrature and dissipative semigroups:
        K(t, s) = sum_{m=1}^M lambda_m * phi_m(t) * psi_m(s) * exp(-gamma_m * (t - s))
    """
    d_model: int = D_MODEL
    n_heads: int = N_HEADS
    m_modes: int = M_MODES
    block_size: int = BLOCK_SIZE
    dtype: jnp.dtype = DTYPE

    def setup(self) -> None:
        self.head_dim = self.d_model // self.n_heads

        # 1. First-kind Chebyshev polynomials T_m(x) on normalized domain [-1, 1]
        t = jnp.arange(self.block_size, dtype=jnp.float32)
        m = jnp.arange(self.m_modes, dtype=jnp.float32)[None, :]
        t_norm = (2.0 * t[:, None] / max(self.block_size - 1, 1)) - 1.0
        t_clipped = jnp.minimum(jnp.maximum(t_norm, -1.0 + 1e-6), 1.0 - 1e-6)
        cheb = jnp.cos(m * jnp.arccos(t_clipped))

        # 2. Gauss-Chebyshev analytic quadrature density weights
        x_s = (2.0 * t + 1.0) / (2.0 * self.block_size)
        w_cheb = 1.0 / jnp.sqrt(x_s * (1.0 - x_s) + 1e-5)
        w_cheb = (w_cheb / jnp.mean(w_cheb))[:, None]

        self.base_phi = cheb
        self.base_psi = cheb * w_cheb

        # 3. Temporal distance delta grid
        t_grid = t[:, None]
        s_grid = t[None, :]
        self.delta_ts = jnp.maximum(t_grid - s_grid, 0.0)
        self.causal_mask = jnp.tril(jnp.ones((self.block_size, self.block_size), dtype=jnp.bool_))

    @nn.compact
    def __call__(self, x: jnp.ndarray, audit: bool = False) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        b, t, c = x.shape

        c_attn = nn.Dense(3 * self.d_model, use_bias=False, dtype=self.dtype, name="c_attn")
        c_proj = nn.Dense(self.d_model, use_bias=False, dtype=self.dtype, name="c_proj")

        lambdas = self.param(
            "lambdas",
            lambda rng: jnp.ones((self.n_heads, self.m_modes), dtype=jnp.float32) / math.sqrt(self.m_modes)
        )
        init_gammas = jnp.broadcast_to(
            jnp.linspace(math.log(0.005), math.log(0.20), self.m_modes),
            (self.n_heads, self.m_modes)
        )
        log_gamma = self.param("log_gamma", lambda rng: init_gammas)

        adapt_phi = nn.Dense(self.m_modes, use_bias=False, kernel_init=eye_initializer, name="adapt_phi")
        adapt_psi = nn.Dense(self.m_modes, use_bias=False, kernel_init=eye_initializer, name="adapt_psi")

        qkv = c_attn(x)
        q, k, v = jnp.split(qkv, 3, axis=-1)

        q = q.reshape((b, t, self.n_heads, self.head_dim)).swapaxes(1, 2)
        k = k.reshape((b, t, self.n_heads, self.head_dim)).swapaxes(1, 2)
        v = v.reshape((b, t, self.n_heads, self.head_dim)).swapaxes(1, 2)

        q_proj = project_conical(q)
        k_proj = project_conical(k)

        phi = adapt_phi(self.base_phi[:t, :])
        psi = adapt_psi(self.base_psi[:t, :])

        # Dissipative Semigroup: exp(-gamma * (t - s))
        gamma = jnp.minimum(jnp.maximum(jnp.exp(log_gamma), 1e-4), 2.0)
        gamma_exp = gamma[:, :, None, None]
        delta = self.delta_ts[:t, :t][None, None, :, :]

        decay_matrix = jnp.exp(-gamma_exp * delta)
        base_kernel = jnp.einsum("tm,sm->mst", phi, psi)[None, :, :, :]

        lam = lambdas[:, :, None, None]
        causal_kernel = jnp.sum(lam * base_kernel * decay_matrix, axis=1)
        causal_kernel = jnp.where(self.causal_mask[:t, :t][None, :, :], causal_kernel, 0.0)

        causal_kernel_bf16 = causal_kernel.astype(self.dtype)

        # Associative Content Integration: [1, H, T, T] @ [B, H, T, D_h]
        input_content = k_proj * v
        integrated_state = jnp.matmul(causal_kernel_bf16[None, :, :, :], input_content)
        state_conical = project_conical(integrated_state)

        out = q_proj * state_conical
        out = out.swapaxes(1, 2).reshape((b, t, c))

        if audit:
            max_val = jnp.max(jnp.abs(causal_kernel))
            k_abs = jnp.abs(causal_kernel)
            probs = k_abs / (jnp.sum(k_abs, axis=-1, keepdims=True) + 1e-7)
            probs = jnp.maximum(probs, 1e-9)
            entropy = -jnp.mean(jnp.sum(probs * jnp.log(probs), axis=-1))
        else:
            max_val = jnp.array(0.0, dtype=jnp.float32)
            entropy = jnp.array(0.0, dtype=jnp.float32)

        return c_proj(out), max_val, entropy


# -----------------------------------------------------------------------------
# 4. Transformer Block and Language Model
# -----------------------------------------------------------------------------
class VolterraTransformerBlock(nn.Module):
    d_model: int = D_MODEL
    n_heads: int = N_HEADS
    m_modes: int = M_MODES
    block_size: int = BLOCK_SIZE
    dtype: jnp.dtype = DTYPE

    @nn.compact
    def __call__(self, x: jnp.ndarray, audit: bool = False) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        norm1 = ConicalNorm(self.d_model, name="norm1")
        norm2 = ConicalNorm(self.d_model, name="norm2")
        attn = ScaledVolterraAttention(
            d_model=self.d_model,
            n_heads=self.n_heads,
            m_modes=self.m_modes,
            block_size=self.block_size,
            dtype=self.dtype,
            name="attn"
        )

        attn_out, max_val, entropy = attn(norm1(x), audit=audit)
        x = x + attn_out

        mlp_fc1 = nn.Dense(4 * self.d_model, use_bias=False, dtype=self.dtype, name="mlp_fc1")
        mlp_fc2 = nn.Dense(self.d_model, use_bias=False, dtype=self.dtype, name="mlp_fc2")
        x = x + mlp_fc2(nn.silu(mlp_fc1(norm2(x))))
        return x, max_val, entropy


class VolterraGPT(nn.Module):
    vocab_size: int = VOCAB_SIZE
    d_model: int = D_MODEL
    n_heads: int = N_HEADS
    n_layers: int = N_LAYERS
    m_modes: int = M_MODES
    block_size: int = BLOCK_SIZE
    dtype: jnp.dtype = DTYPE

    @nn.compact
    def __call__(
        self,
        idx: jnp.ndarray,
        targets: jnp.ndarray = None,
        audit: bool = False
    ) -> Tuple[jnp.ndarray, Any, jnp.ndarray, jnp.ndarray]:
        b, t = idx.shape

        tok_embed = self.param(
            "token_emb",
            nn.initializers.normal(stddev=0.02),
            (self.vocab_size, self.d_model),
            self.dtype
        )
        pos_embed = self.param(
            "pos_emb",
            nn.initializers.normal(stddev=0.02),
            (self.block_size, self.d_model),
            self.dtype
        )

        pos = jnp.arange(t)[None, :]
        x = tok_embed[idx] + pos_embed[pos]

        max_vals, entropies = [], []
        for i in range(self.n_layers):
            x, mv, ent = VolterraTransformerBlock(
                d_model=self.d_model,
                n_heads=self.n_heads,
                m_modes=self.m_modes,
                block_size=self.block_size,
                dtype=self.dtype,
                name=f"block_{i}"
            )(x, audit=audit)
            if audit:
                max_vals.append(mv)
                entropies.append(ent)

        x = ConicalNorm(self.d_model, name="norm_final")(x)
        logits = jnp.matmul(x, tok_embed.T)

        loss = None
        if targets is not None:
            loss = optax.softmax_cross_entropy_with_integer_labels(
                logits=logits.astype(jnp.float32),
                labels=targets
            ).mean()

        if audit:
            peak_val = jnp.max(jnp.stack(max_vals))
            avg_ent = jnp.mean(jnp.stack(entropies))
        else:
            peak_val = jnp.array(0.0, dtype=jnp.float32)
            avg_ent = jnp.array(0.0, dtype=jnp.float32)

        return logits, loss, peak_val, avg_ent


# -----------------------------------------------------------------------------
# 5. Compiled Optimization Routines
# -----------------------------------------------------------------------------
def create_train_state(rng: Any, model: nn.Module, lr: float) -> train_state.TrainState:
    dummy_x = jnp.ones((BATCH_SIZE, BLOCK_SIZE), dtype=jnp.int32)
    params = model.init(rng, dummy_x, targets=dummy_x)["params"]

    schedule = optax.cosine_decay_schedule(init_value=lr, decay_steps=TOTAL_STEPS)
    tx = optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(learning_rate=schedule, weight_decay=WEIGHT_DECAY, b1=0.9, b2=0.95)
    )
    return train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)


@jax.jit
def train_step(state: train_state.TrainState, x: jnp.ndarray, y: jnp.ndarray) -> Tuple[train_state.TrainState, jnp.ndarray]:
    def loss_fn(params):
        _, loss, _, _ = state.apply_fn({"params": params}, x, targets=y, audit=False)
        return loss

    grad_fn = jax.value_and_grad(loss_fn)
    loss, grads = grad_fn(state.params)
    new_state = state.apply_gradients(grads=grads)
    return new_state, loss


@jax.jit
def eval_step(state: train_state.TrainState, x: jnp.ndarray, y: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    _, loss, max_val, entropy = state.apply_fn({"params": state.params}, x, targets=y, audit=True)
    return loss, max_val, entropy


def evaluate(state: train_state.TrainState, iters: int = EVAL_ITERS) -> Tuple[float, float, float, float]:
    losses, max_vals, entropies = [], [], []
    for _ in range(iters):
        xb, yb = get_batch(VAL_DATA)
        loss, mv, ent = eval_step(state, xb, yb)
        losses.append(float(loss))
        max_vals.append(float(mv))
        entropies.append(float(ent))

    val_loss = float(np.mean(losses))
    ppl = math.exp(min(val_loss, 15.0))
    peak_val = float(np.max(max_vals))
    mean_ent = float(np.mean(entropies))
    return val_loss, ppl, peak_val, mean_ent


# -----------------------------------------------------------------------------
# 6. Benchmark Execution Loop
# -----------------------------------------------------------------------------
def run_benchmark() -> None:
    print("=" * 115)
    print(f"[INFO] Device: {DEVICES[0].device_kind.upper()} | Backend: {jax.default_backend()} | Precision: bfloat16")
    print(f"[INFO] Tokens: Train={len(TRAIN_DATA):,} | Validation={len(VAL_DATA):,} | Vocab={VOCAB_SIZE}")
    print(f"[INFO] Steps: {TOTAL_STEPS} | Batch: {BATCH_SIZE} | Context: {BLOCK_SIZE} | Modes: {M_MODES}")
    print("=" * 115)

    rng = random.PRNGKey(SEED)
    rng, init_rng = random.split(rng)

    model = VolterraGPT(
        vocab_size=VOCAB_SIZE,
        d_model=D_MODEL,
        n_heads=N_HEADS,
        n_layers=N_LAYERS,
        m_modes=M_MODES,
        block_size=BLOCK_SIZE,
        dtype=DTYPE,
    )

    state = create_train_state(init_rng, model, LR)
    total_params = sum(x.size for x in jax.tree_util.tree_leaves(state.params))
    print(f"[INFO] Initialized Model: Volterra-Fundamental | Parameters: {total_params / 1e6:.1f}M")

    print("[INFO] Compiling XLA computation graphs...")
    t_compile = time.time()
    dummy_x, dummy_y = get_batch(TRAIN_DATA)
    state, _ = train_step(state, dummy_x, dummy_y)
    _ = eval_step(state, dummy_x, dummy_y)
    jax.block_until_ready(state.params)
    print(f"[INFO] Compilation completed in {time.time() - t_compile:.2f}s")

    t0 = time.time()

    for step in range(1, TOTAL_STEPS + 1):
        xb, yb = get_batch(TRAIN_DATA)
        state, loss = train_step(state, xb, yb)

        if step % EVAL_INTERVAL == 0 or step == TOTAL_STEPS:
            jax.block_until_ready(loss)
            v_loss, v_ppl, v_val, v_ent = evaluate(state)
            throughput = (step * BATCH_SIZE * BLOCK_SIZE) / (time.time() - t0)
            print(
                f"  Step {step:>4}/{TOTAL_STEPS} | Val Loss: {v_loss:.4f} | PPL: {v_ppl:>6.2f} | "
                f"Entropy: {v_ent:.3f} | Kernel Max: {v_val:>5.1f} | Throughput: {throughput / 1e3:>5.1f}k tok/s"
            )

    elapsed = time.time() - t0
    final_loss, final_ppl, final_val, final_ent = evaluate(state, iters=80)
    final_throughput = (TOTAL_STEPS * BATCH_SIZE * BLOCK_SIZE) / elapsed

    print("\n" + "=" * 115)
    print("CONSOLIDATED EMPIRICAL SUMMARY: VOLTERRA-FUNDAMENTAL (JAX XLA / A100)")
    print("=" * 115)
    print(f"{'ARCHITECTURE':<26} | {'VAL LOSS':<12} | {'PPL':<10} | {'ENTROPY':<10} | {'KERNEL MAX':<12} | {'THROUGHPUT':<14} | {'TIME'}")
    print("-" * 115)
    print(
        f"{'Volterra JAX (M=16)':<26} | {final_loss:>8.4f}   | {final_ppl:>6.2f}   | "
        f"{final_ent:>7.3f}  | {final_val:>8.1f}     | {final_throughput / 1e3:>5.1f}k tok/s  | {elapsed:>5.1f}s"
    )
    print("=" * 115)


if __name__ == "__main__":
    run_benchmark()

[INFO] Device: NVIDIA A100-SXM4-80GB | Backend: gpu | Precision: bfloat16
[INFO] Tokens: Train=2,448,382 | Validation=258,659 | Vocab=50257
[INFO] Steps: 1000 | Batch: 32 | Context: 512 | Modes: 16
[INFO] Initialized Model: Volterra-Fundamental | Parameters: 51.2M
[INFO] Compiling XLA computation graphs...
[INFO] Compilation completed in 29.36s
  Step   50/1000 | Val Loss: 5.5463 | PPL: 256.29 | Entropy: 4.656 | Kernel Max:  42.5 | Throughput: 234.4k tok/s
  Step  100/1000 | Val Loss: 5.1674 | PPL: 175.47 | Entropy: 4.426 | Kernel Max:  42.2 | Throughput: 232.2k tok/s
  Step  150/1000 | Val Loss: 4.9689 | PPL: 143.87 | Entropy: 4.233 | Kernel Max:  42.0 | Throughput: 230.9k tok/s
  Step  200/1000 | Val Loss: 4.7771 | PPL: 118.76 | Entropy: 4.101 | Kernel Max:  42.2 | Throughput: 229.8k tok/s
  Step  250/1000 | Val Loss: 4.6864 | PPL: 108.47 | Entropy: 4.047 | Kernel Max:  42.4 | Throughput: 229.4k tok/s
  Step  300/1000 | Val Loss: 4.5626 | PPL:  95.83 | Entropy: 4.017 | Kernel Max:  4